# **BYTE PAIR ENCODING (BPE) - TOKENIZER**


---



## 1. **Character-level Tokenization:**

65 unique characters, each mapped to an integer. `stoi`/`itos`, `encode()`/`decode()` — simple.

But character-level tokenization has a real ceiling:

- Every character is a separate prediction. The word "Shakespeare" is 11 separate tokens the model has to get right in sequence.
- The model spends its limited capacity (128-dim embeddings, 4 layers) learning to spell common words letter-by-letter, instead of learning *meaning* and *structure*.
- Your context window (`block_size = 64`) only covers ~64 characters — roughly 10-12 words. A subword tokenizer would let the same 64 "slots" cover 3-4x more actual text.

## 2. What **Subword Tokenization** actually is

The core idea: don't tokenize by character (too fine, too many tokens) and don't tokenize by whole word (too coarse, vocabulary explodes — every conjugation, typo, and rare word needs its own slot). Split the difference: tokenize by **frequently-occurring chunks of characters**.

Concretely, a real BPE tokenizer would encode:
- `"the"` → one token (extremely common, gets its own slot)
- `"tokenization"` → maybe `["token", "ization"]` (two tokens — common suffix "ization" is shared across thousands of words)
- `"Grzegorzewski"` (a name it's never seen) → falls back to something like `["G", "rz", "ego", "rz", "ew", "ski"]` — never *fails*, just degrades gracefully to smaller pieces.

## 3. **Byte Pair Encoding** — the algorithm behind it

BPE is how you *build* that vocabulary of chunks. The algorithm, from scratch:

1. **Start at the character level.** Every character is its own token. (Sound familiar? This is literally your current setup — BPE starts exactly where MiniGPT already is.)
2. **Count every adjacent pair of tokens** in your training text. E.g., in "the cat sat", pairs are `(t,h)`, `(h,e)`, `(e,space)`, `(space,c)`, `(c,a)`, `(a,t)`, ...
3. **Find the single most frequent pair** across the whole corpus. Say `(t, h)` appears 10,000 times — more than any other pair.
4. **Merge that pair into one new token.** Every `t` immediately followed by `h` becomes a single unit `th`. Vocabulary grows by 1 (now has `th` in addition to `t` and `h`).
5. **Repeat.** Recount pairs (now including pairs involving your new `th` token, e.g., `(th, e)`), find the new most frequent pair, merge it. Maybe `th`+`e` → `the` becomes its own token next.
6. **Stop after N merges** — N is a hyperparameter you choose upfront. It directly determines your final vocab size: `final_vocab_size = starting_chars + N merges`.

So if you start with 65 characters and do, say, 1,935 merges, you land on a 2,000-token vocabulary. Every merge you perform makes one more "chunk" a single token instead of multiple characters — that's literally where the compression comes from.

---

## **What we're building**
A from-scratch BPE trainer — it takes your corpus (Tiny Shakespeare) and runs the merge loop from last session (count pairs → merge most frequent → repeat) for N iterations, producing your own vocabulary and merge rules. Then an encoder/decoder that applies those learned merges to any new text.

We start from bytes, not characters — same trick GPT-2 uses. `text.encode("utf-8")` turns any string into raw bytes `0-255`. This means our starting vocabulary is always exactly 256 tokens.


---



### **Step 1**: Split on Whitespaces



In [ ]:
import re

def split_on_whitespace(text):
    """
    Splits text into a list of chunks, where every chunk is either
    a whitespace run or a whitespace-free stretch of characters.
    GPT-2 does something similar with a regex; this is the simple version.

    e.g. "Once upon a time" -> ["Once", " ", "upon", " ", "a", " ", "time"]
    """
    return re.findall(r'\s+|\S+', text)

### **Step 2**: represent text as a list of tokens, byte-level

In [ ]:
def get_stats(ids):
    """
    Count how often every adjacent pair occurs.
    ids: list of ints (current token sequence)
    Returns: dict {(id1, id2): count}
    """
    counts = {}
    for pair in zip(ids, ids[1:]):          # (ids[0],ids[1]), (ids[1],ids[2]), ...
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    """
    Replace every occurrence of `pair` in ids with a single new_id.
    ids: list of ints
    pair: (id1, id2) tuple to merge
    new_id: the new token id to substitute
    """
    new_ids = []
    i = 0
    while i < len(ids):
        # Check if this pair matches at position i
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2                           # skip both merged tokens
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

In [ ]:
def get_stats_multi(chunks, counts=None):
    """
    Same as get_stats, but counts pairs WITHIN each chunk only —
    never across chunk boundaries. This is what enforces the
    "no merge crosses a word boundary" rule.

    chunks: list of token-id lists, one per whitespace-delimited chunk
    """
    counts = {} if counts is None else counts
    for chunk_ids in chunks:
        for pair in zip(chunk_ids, chunk_ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
    return counts

In [ ]:
def merge_multi(chunks, pair, new_id):
    """
    Same as merge, but applies within each chunk independently.
    """
    return [merge(chunk_ids, pair, new_id) for chunk_ids in chunks]

### **Step 2** — the training loop


In [ ]:
def train_bpe(text, vocab_size=None, min_frequency=10):
    """
    vocab_size    : soft upper bound on merges (None = no cap, rely purely on min_frequency)
    min_frequency : stop merging once the best remaining pair occurs fewer
                    than this many times — implements idea #4
    """
    chunks = [list(chunk.encode("utf-8")) for chunk in split_on_whitespace(text)]

    merges = {}
    i = 0
    while True:
        if vocab_size is not None and 256 + i >= vocab_size:
            print(f"Stopped: reached target vocab_size={vocab_size}")
            break

        stats = get_stats_multi(chunks)
        if not stats:
            print("Stopped: no pairs left to merge")
            break

        pair = max(stats, key=stats.get)
        best_count = stats[pair]

        if best_count < min_frequency:
            print(f"Stopped: best remaining pair {pair} only occurs {best_count} times (< {min_frequency})")
            break

        new_id = 256 + i
        chunks = merge_multi(chunks, pair, new_id)
        merges[pair] = new_id

        if i % 100 == 0:
            print(f"merge {i}: {pair} -> {new_id} (had {best_count} occurrences)")

        i += 1

    return merges, chunks

### **Step 3** — encode and decode using the learned merges

In [ ]:
def bpe_encode(text, merges):
    """
    Encodes text using whitespace-respecting merges.
    Splits into chunks first, encodes each chunk independently,
    then concatenates the results.
    """
    chunks = split_on_whitespace(text)
    ids = []
    for chunk in chunks:
        chunk_ids = list(chunk.encode("utf-8"))
        while len(chunk_ids) >= 2:
            stats = get_stats(chunk_ids)   # single-chunk version — correct here
            pair = min(stats, key=lambda p: merges.get(p, float("inf")))
            if pair not in merges:
                break
            chunk_ids = merge(chunk_ids, pair, merges[pair])
        ids.extend(chunk_ids)
    return ids

def bpe_decode(ids, merges):
    """
    Reverse the merges to get back raw bytes, then decode to text.
    """
    # Build id -> bytes mapping, starting from the 256 raw bytes
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for (p0, p1), idx in merges.items():
        vocab[idx] = vocab[p0] + vocab[p1]   # concatenate the two pieces

    tokens = b"".join(vocab[idx] for idx in ids)
    return tokens.decode("utf-8", errors="replace")

### **Step 4** — Wiring it into MiniGPT

In [ ]:
from datasets import load_dataset

# Streaming avoids downloading the full 2.1M-story dataset
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

# Subsample — 15,000 stories is plenty of repetitive structure
# without blowing past Colab free-tier session limits
num_stories = 500
stories = []
for i, example in enumerate(ds):
    if i >= num_stories:
        break
    stories.append(example["text"])

text = "\n".join(stories)
print(f"Total characters: {len(text):,}")
print(f"First 300 chars:\n{text[:300]}")

Total characters: 402,933
First 300 chars:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and 


In [ ]:
# Train once, on any dataset
merges, _ = train_bpe(text, min_frequency=100)   # pick your target vocab size or minimum frequency
vocab_size = 256 + len(merges)

encode = lambda s: bpe_encode(s, merges)
decode = lambda l: bpe_decode(l, merges)

# Test it
sample = "To be or not to be"
tokens = encode(sample)
print(f"Tokens: {tokens}")
print(f"Decoded: {decode(tokens)}")   # should exactly reconstruct the original

merge 0: (104, 101) -> 256 (had 11727 occurrences)
merge 100: (258, 121) -> 356 (had 434 occurrences)
merge 200: (282, 274) -> 456 (had 202 occurrences)
merge 300: (112, 97) -> 556 (had 129 occurrences)
Stopped: best remaining pair (263, 114) only occurs 99 times (< 100)
Tokens: [84, 111, 32, 288, 32, 290, 32, 595, 32, 262, 32, 288]
Decoded: To be or not to be


In [ ]:
# print(list(merges.items())[1600:1744])

In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print(f"Total merges learned: {len(merges)}")
print(f"Final vocab size: {256 + len(merges)}\n")

# Look at the tail, whatever size it ended up being
tail_size = 150
for (p0, p1), idx in list(merges.items())[-tail_size:]:
    piece = vocab[idx].decode("utf-8", errors="replace")
    print(f"{idx}: {repr(piece)}  (merged from {p0},{p1})")

# The actual correctness check: flag anything that looks like a phrase-crossing token
print("\n--- Checking for any boundary violations ---")
violations = 0
for idx, piece_bytes in vocab.items():
    if idx < 256:
        continue
    piece = piece_bytes.decode("utf-8", errors="replace")
    stripped = piece.strip()
    if " " in stripped:   # a space survives even after stripping leading/trailing = it's in the middle
        print(f"VIOLATION — token {idx}: {repr(piece)}")
        violations += 1

print(f"\n{violations} boundary violations found out of {len(merges)} merges")

Total merges learned: 367
Final vocab size: 623

473: 'spe'  (merged from 385,101)
474: 'uc'  (merged from 117,99)
475: 'ong'  (merged from 272,103)
476: 'ud'  (merged from 117,100)
477: 'own'  (merged from 301,110)
478: 'get'  (merged from 338,116)
479: 'then'  (merged from 258,110)
480: 'gether'  (merged from 338,367)
481: 'man'  (merged from 109,257)
482: '?"'  (merged from 63,34)
483: 'k.'  (merged from 107,46)
484: 'it.'  (merged from 265,46)
485: 'felt'  (merged from 343,460)
486: 'ave'  (merged from 97,319)
487: 'qu'  (merged from 113,117)
488: 'have'  (merged from 269,319)
489: 'fun'  (merged from 102,333)
490: 'old'  (merged from 111,289)
491: 'when'  (merged from 119,415)
492: 'So'  (merged from 83,111)
493: 'friends'  (merged from 384,115)
494: 'wal'  (merged from 263,108)
495: 'home'  (merged from 325,275)
496: 'pped'  (merged from 312,259)
497: 'But'  (merged from 66,317)
498: 'sto'  (merged from 115,262)
499: 'ree'  (merged from 264,101)
500: 'every'  (merged from 101,309